In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import datetime
import json

CATALOG = "olist_ecommerce"
RUN_ID  = datetime.now().strftime("%Y%m%d_%H%M%S")

# ── Structured Logger ─────────────────────────────────────────
# Using structured logging instead of plain print()
# → Easier to parse, monitor, and debug in production
dq_log = []  # Store all DQ results for end-of-pipeline summary

def log(level, table, message, metrics=None):
    """
    Structured logger — production standard.
    level: INFO / WARN / CRITICAL / SUCCESS
    """
    entry = {
        "run_id":    RUN_ID,
        "timestamp": datetime.now().strftime("%H:%M:%S"),
        "level":     level,
        "table":     table,
        "message":   message,
        "metrics":   metrics or {}
    }
    dq_log.append(entry)

    icon = {"INFO":"ℹ", "WARN":"⚠", "CRITICAL":"✖", "SUCCESS":"✔"}.get(level, "·")
    metric_str = " | ".join([f"{k}={v}" for k,v in (metrics or {}).items()])
    print(f"[{entry['timestamp']}] {icon} {level:<8} {table:<25} {message}"
          + (f"  ({metric_str})" if metric_str else ""))

print(f"Silver transform started — Run ID: {RUN_ID}")
log("INFO", "system", "Silver layer initialized", {"catalog": CATALOG})

In [0]:
def dq_check(df, table_name, critical_rules, warn_rules=[],
             dedup_keys=None):
    print(f"\n{'─'*55}")
    print(f"DQ CHECK: {table_name}")
    print(f"{'─'*55}")

    initial_count = df.count()
    result = df
    table_metrics = {
        "table":            table_name,
        "initial_rows":     initial_count,
        "critical_dropped": 0,
        "warn_violations":  0,
        "final_rows":       0,
        "drop_rate_pct":    "0.00"
    }

    log("INFO", table_name, "Starting DQ",
        {"input_rows": f"{initial_count:,}"})

    # ── Deduplication ─────────────────────────────────────────
    if dedup_keys:
        before = result.count()
        result = result.dropDuplicates(dedup_keys)
        dupes  = before - result.count()
        if dupes > 0:
            log("WARN", table_name,
                f"Removed {dupes:,} duplicate rows",
                {"keys": str(dedup_keys), "dupes": str(dupes)})
        else:
            log("INFO", table_name, "No duplicates found")

    # ── Critical Rules: DROP ──────────────────────────────────
    total_dropped = 0
    if critical_rules:
        print("\n  [Critical rules — violating rows DROPPED]")

    for col_name, condition, description in critical_rules:
        before     = result.count()
        violating  = result.filter(f"NOT ({condition})")
        viol_count = violating.count()

        if viol_count > 0:
            result    = result.filter(condition)
            dropped   = before - result.count()
            rate_pct  = f"{dropped / before * 100:.2f}"
            total_dropped += dropped

            log("CRITICAL", table_name,
                f"DROP — {col_name}: {description}",
                {"dropped": f"{dropped:,}", "rate": f"{rate_pct}%"})

            print(f"\n    Sample vi phạm ({col_name}):")
            try:
                if col_name in violating.columns:
                    violating.select(col(col_name)).limit(3).show(truncate=False)
                else:
                    print(f"    (column {col_name} not found)")
            except Exception as e:
                print(f"    (could not show sample: {str(e)[:60]})")
        else:
            print(f"  ✔ {col_name}: OK — {description}")

    table_metrics["critical_dropped"] = total_dropped

    # ── Warn Rules: LOG only ──────────────────────────────────
    if warn_rules:
        print("\n  [Warn rules — violations LOGGED only]")

    for col_name, condition, description in warn_rules:
        violations = result.filter(f"NOT ({condition})").count()
        if violations > 0:
            pct = f"{violations / result.count() * 100:.2f}"
            log("WARN", table_name,
                f"WARN — {col_name}: {description}",
                {"violations": f"{violations:,}", "pct": f"{pct}%"})
            table_metrics["warn_violations"] += violations
        else:
            print(f"  ✔ {col_name}: OK — {description}")

    # ── Summary ───────────────────────────────────────────────
    final_count = result.count()
    drop_rate   = f"{total_dropped / initial_count * 100:.2f}" \
                  if initial_count > 0 else "0.00"

    table_metrics["final_rows"]    = final_count
    table_metrics["drop_rate_pct"] = drop_rate
    dq_log.append(table_metrics)

    log("SUCCESS", table_name, "DQ complete",
        {"in":        f"{initial_count:,}",
         "out":       f"{final_count:,}",
         "dropped":   f"{total_dropped:,}",
         "drop_rate": f"{drop_rate}%"})

    return result


def write_silver(df, table_name, partition_cols=None, zorder_cols=None):
    """Write Silver table — simplified for Free Edition."""
    full_table = f"{CATALOG}.silver.{table_name}"

    writer = (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true"))

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    writer.saveAsTable(full_table)

    count = spark.read.table(full_table).count()
    log("SUCCESS", table_name,
        f"Written to {full_table}",
        {"rows": f"{count:,}"})

In [0]:
"""
BUSINESS RULES cho silver_orders:
─────────────────────────────────
R1: order_purchase_timestamp KHÔNG được null
    → Không có timestamp = không biết khi nào đặt hàng = vô nghĩa
R2: delivered_date null → fill bằng estimated_date
    → Orders chưa giao chưa có delivered_date
    → Dùng estimated để tính lead_time gần đúng
    → Đánh dấu bằng is_delivery_date_estimated=True để phân biệt
R3: lead_time âm → giữ lại nhưng warn
    → Có thể do timezone issue hoặc data entry error
    → Không drop vì vẫn muốn phân tích
R4: order_status phải nằm trong whitelist
    → Prevent garbage values từ source
"""

log("INFO", "silver_orders", "Starting transform")
bronze_orders = spark.read.table(f"{CATALOG}.bronze.orders")

silver_orders = (bronze_orders

    # ── STEP 1: Parse timestamps ──────────────────────────────
    # Bronze: tất cả String vì đọc từ CSV
    # Silver: phải là Timestamp để tính toán được
    .withColumn("order_purchase_timestamp",
        to_timestamp("order_purchase_timestamp"))
    .withColumn("order_approved_at",
        to_timestamp("order_approved_at"))
    .withColumn("order_delivered_carrier_date",
        to_timestamp("order_delivered_carrier_date"))
    .withColumn("order_delivered_customer_date",
        to_timestamp("order_delivered_customer_date"))
    .withColumn("order_estimated_delivery_date",
        to_timestamp("order_estimated_delivery_date"))

    # ── STEP 2: Null Strategy — documented business rule ──────
    # Flag TRƯỚC khi fill để biết row nào bị fill
    # Quan trọng: đừng bao giờ fill null mà không đánh dấu
    .withColumn("is_delivery_date_estimated",
        col("order_delivered_customer_date").isNull() &
        col("order_status").isin(
            "shipped", "invoiced", "processing",
            "created", "approved"))

    # Fill null delivered_date bằng estimated
    # Business rule R2: documented rõ ràng
    .withColumn("order_delivered_customer_date",
        coalesce(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")))

    # ── STEP 3: Core KPIs ─────────────────────────────────────
    # KPI 1: Lead time (ngày từ đặt hàng đến nhận hàng)
    # Đây là SLA metric quan trọng nhất của e-commerce
    .withColumn("lead_time_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")))

    # KPI 2: Approval time (giờ từ đặt đến seller confirm)
    # Measure seller responsiveness
    .withColumn("approval_time_hours",
    ((unix_timestamp("order_approved_at") -
      unix_timestamp("order_purchase_timestamp"))
     / 3600).cast("double"))

    # KPI 3: Carrier time (ngày từ kho đến tay khách)
    # Đo logistics performance
    .withColumn("carrier_time_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_delivered_carrier_date")))

    # KPI 4: Days before estimated (âm = trễ, dương = sớm)
    # Measure: giao sớm/trễ bao nhiêu ngày so với hứa
    .withColumn("days_before_estimated",
        datediff(
            col("order_estimated_delivery_date"),
            col("order_delivered_customer_date")))

    # KPI 5: On-time flag
    .withColumn("is_delivered_on_time",
    when(col("is_delivery_date_estimated") == True,
         lit(None).cast("boolean"))
    .otherwise(
        col("order_delivered_customer_date") <=
        col("order_estimated_delivery_date")))

    # ── STEP 4: Delivery tier ─────────────────────────────────
    # Bucket lead_time thành tier để GROUP BY dễ hơn
    .withColumn("delivery_speed_tier",
        when(col("lead_time_days") <= 3,  "express")
        .when(col("lead_time_days") <= 7,  "fast")
        .when(col("lead_time_days") <= 14, "standard")
        .when(col("lead_time_days") <= 30, "slow")
        .when(col("lead_time_days").isNotNull(), "very_slow")
        .otherwise("unknown"))

    # ── STEP 5: Time dimensions ───────────────────────────────
    # Pre-compute để GROUP BY nhanh hơn ở Business layer
    .withColumn("purchase_year",
        year("order_purchase_timestamp"))
    .withColumn("purchase_month",
        month("order_purchase_timestamp"))
    .withColumn("purchase_quarter",
        quarter("order_purchase_timestamp"))
    .withColumn("purchase_day_of_week",
        date_format("order_purchase_timestamp", "EEEE"))
    .withColumn("is_weekend_purchase",
        dayofweek("order_purchase_timestamp").isin([1, 7]))
    .withColumn("purchase_hour",
        hour("order_purchase_timestamp"))

    # ── STEP 6: Normalize ─────────────────────────────────────
    .withColumn("order_status",
        lower(trim(col("order_status"))))

    # ── STEP 7: Drop Bronze metadata ──────────────────────────
    .drop("_source_file", "_batch_id")
)

# ── DQ Checks ─────────────────────────────────────────────────
silver_orders = dq_check(
    silver_orders, "silver_orders",
    critical_rules=[
        ("order_id",
         "order_id IS NOT NULL",
         "Primary key không được null"),
        ("customer_id",
         "customer_id IS NOT NULL",
         "Foreign key không được null"),
        ("order_purchase_timestamp",
         "order_purchase_timestamp IS NOT NULL",
         "Business rule R1: không có timestamp = vô nghĩa"),
        ("order_status",
         "order_status IN ('delivered','shipped','canceled',"
         "'unavailable','invoiced','processing',"
         "'created','approved')",
         "Status phải nằm trong whitelist"),
    ],
    warn_rules=[
        ("lead_time_days",
         "lead_time_days >= 0",
         "Lead time âm: có thể do timezone issue"),
        ("lead_time_days",
         "lead_time_days <= 365",
         "Lead time > 1 năm: nghi ngờ data lỗi"),
        ("approval_time_hours",
         "approval_time_hours >= 0",
         "Approval trước khi đặt hàng: suspicious"),
    ]
)

# ── Write đơn giản — thay thế write_silver() ──────────────────
(silver_orders.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.orders"))

count = spark.read.table(f"{CATALOG}.silver.orders").count()
log("SUCCESS", "silver_orders",
    f"Written to {CATALOG}.silver.orders",
    {"rows": f"{count:,}"})

# ── Business insight check ─────────────────────────────────────
print("\n=== KPI Overview ===")
spark.sql(f"""
    SELECT
        order_status,
        COUNT(*)                             AS total,
        ROUND(AVG(lead_time_days), 1)        AS avg_lead_days,
        ROUND(AVG(approval_time_hours), 1)   AS avg_approval_hrs,
        ROUND(AVG(days_before_estimated), 1) AS avg_days_early,
        SUM(CASE WHEN is_delivered_on_time
            THEN 1 ELSE 0 END)               AS on_time,
        COUNT(CASE WHEN delivery_speed_tier = 'express'
            THEN 1 END)                      AS express_count
    FROM {CATALOG}.silver.orders
    GROUP BY order_status
    ORDER BY total DESC
""").show()

In [0]:
log("INFO", "silver_order_items", "Starting transform")
bronze_order_items = spark.read.table(f"{CATALOG}.bronze.order_items")
silver_order_items = (bronze_order_items
    # Cast đúng kiểu
    .withColumn("order_item_id",
        col("order_item_id").cast("int"))
    .withColumn("price",
        col("price").cast("double"))
    .withColumn("freight_value",
        col("freight_value").cast("double"))
    .withColumn("shipping_limit_date",
        to_timestamp("shipping_limit_date"))
     
     # Derived columns
    .withColumn("total_item_value",
        col("price") + col("freight_value"))
    .withColumn("freight_ratio",
        (col("freight_value") /
         when(col("price") == 0, lit(1.0))
         .otherwise(col("price"))).cast("double"))
    .withColumn("price_tier",
        when(col("price") < 50,   "budget")
        .when(col("price") < 200, "mid_range")
        .when(col("price") < 500, "premium")
        .otherwise("luxury"))

    .drop("_source_file", "_batch_id")
)

silver_order_items = dq_check(
    silver_order_items, "silver_order_items",
    critical_rules=[
        ("order_id",   "order_id IS NOT NULL",   "FK to orders"),
        ("product_id", "product_id IS NOT NULL",  "FK to products"),
        ("seller_id",  "seller_id IS NOT NULL",   "FK to sellers"),
        ("price",      "price > 0",               "Giá phải dương"),
    ],
    warn_rules=[
        ("freight_value",
         "freight_value >= 0",
         "Phí ship không âm"),
        ("freight_ratio",
         "freight_ratio <= 2.0",
         "Phí ship > 2x giá — nghi ngờ"),
    ],
    dedup_keys=["order_id", "order_item_id"]
)

(silver_order_items.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.order_items"))

count = spark.read.table(f"{CATALOG}.silver.order_items").count()
log("SUCCESS", "silver_order_items",
    f"Written to {CATALOG}.silver.order_items",
    {"rows": f"{count:,}"})

# Quick insight
print("\n=== Price tier distribution ===")
spark.sql(f"""
    SELECT price_tier,
           COUNT(*)                     AS items,
           ROUND(AVG(price), 2)         AS avg_price,
           ROUND(AVG(freight_ratio), 3) AS avg_freight_ratio
    FROM {CATALOG}.silver.order_items
    GROUP BY price_tier
    ORDER BY avg_price
""").show()

In [0]:
from pyspark.sql import functions as F

log("INFO", "silver_payments", "Starting transform")
bronze_payments = spark.read.table(f"{CATALOG}.bronze.payments")

silver_payments = (bronze_payments
    .withColumn("payment_value",
        col("payment_value").cast("double"))
    .withColumn("payment_installments",
        col("payment_installments").cast("int"))
    .withColumn("payment_type",
        lower(trim(col("payment_type"))))
    .withColumn("has_installments",
        col("payment_installments") > 1)
    .withColumn("installment_tier",
        when(col("payment_installments") == 1,   "cash")
        .when(col("payment_installments") <= 3,  "short")
        .when(col("payment_installments") <= 12, "medium")
        .otherwise("long"))
    .drop("_source_file", "_batch_id")
)

silver_payments = dq_check(
    silver_payments, "silver_payments",
    critical_rules=[
        ("order_id",
         "order_id IS NOT NULL",
         "FK to orders"),
        ("payment_type",
         "payment_type IN ('credit_card','boleto',"
         "'voucher','debit_card','not_defined')",
         "Type whitelist"),
        # ← Bỏ payment_value > 0 khỏi critical
    ],
    warn_rules=[
        ("payment_value",
         "payment_value >= 0",
         "Payment âm: impossible — data lỗi"),
        ("payment_installments",
         "payment_installments BETWEEN 1 AND 24",
         "Trả góp hợp lý 1-24 kỳ"),
    ],
    dedup_keys=["order_id", "payment_sequential"]
)

# Aggregate 1 row per order
payment_agg = (silver_payments
    .groupBy("order_id")
    .agg(
        F.sum("payment_value").alias("total_payment_value"),
        F.first("payment_type").alias("primary_payment_type"),
        F.max("payment_installments").alias("max_installments"),
        F.count("payment_sequential").alias("payment_count"),
        F.sum(when(col("payment_type") == "credit_card",
            col("payment_value")).otherwise(lit(0.0)))
            .alias("credit_card_value"),
        F.sum(when(col("payment_type") == "boleto",
            col("payment_value")).otherwise(lit(0.0)))
            .alias("boleto_value"),
        F.sum(when(col("payment_type") == "voucher",
            col("payment_value")).otherwise(lit(0.0)))
            .alias("voucher_value"),
        (F.count("payment_sequential") > 1)
            .alias("is_mixed_payment"),
    )
)

(silver_payments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.payments"))

(payment_agg.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.payment_agg"))

log("SUCCESS", "silver_payments",
    "Written payments + payment_agg",
    {"payments": f"{silver_payments.count():,}",
     "agg":      f"{payment_agg.count():,}"})

print("\n=== Payment type distribution ===")
spark.sql(f"""
    SELECT payment_type,
           COUNT(*)                           AS count,
           ROUND(AVG(payment_value), 2)       AS avg_value,
           ROUND(AVG(payment_installments),1) AS avg_installments
    FROM {CATALOG}.silver.payments
    GROUP BY payment_type
    ORDER BY count DESC
""").show()

In [0]:
log("INFO", "silver_reviews", "Starting transformation")
bronze_reviews = spark.read.table(f"{CATALOG}.bronze.reviews")

silver_reviews = (bronze_reviews

    # TRY_CAST — an toàn, không crash khi gặp timestamp string
    .withColumn("review_score",
        expr("TRY_CAST(review_score AS INT)"))

    .withColumn("review_creation_date",
        to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp",
        to_timestamp("review_answer_timestamp"))

    .withColumn("response_time_hours",
        ((unix_timestamp(col("review_answer_timestamp")) -
          unix_timestamp(col("review_creation_date")))
         / lit(3600)).cast("double"))

    .withColumn("response_time_hours",
        when(col("response_time_hours") < 0, lit(None).cast("double"))
        .otherwise(col("response_time_hours")))

    .withColumn("sentiment",
        when(col("review_score") <= 2,        lit("negative"))
        .when(col("review_score") == 3,        lit("neutral"))
        .when(col("review_score").isNull(),    lit("unknown"))
        .otherwise(                            lit("positive")))

    .withColumn("has_comment",
        col("review_comment_message").isNotNull() &
        (length(trim(col("review_comment_message"))) > 0))

    .withColumn("has_title",
        col("review_comment_title").isNotNull() &
        (length(trim(col("review_comment_title"))) > 0))

    .withColumn("review_completeness",
        col("has_comment").cast("int") +
        col("has_title").cast("int") +
        when(col("review_score") >= 4, lit(1)).otherwise(lit(0)))

    .withColumn("response_speed_tier",
        when(col("response_time_hours").isNull(), lit("no_response"))
        .when(col("response_time_hours") <= 24,   lit("fast"))
        .when(col("response_time_hours") <= 72,   lit("normal"))
        .otherwise(                               lit("slow")))

    .drop("_source_file", "_batch_id")
)

silver_reviews = dq_check(
    silver_reviews, "silver_reviews",
    critical_rules=[
        ("review_id",
         "review_id IS NOT NULL",
         "Primary key"),
        ("order_id",
         "order_id IS NOT NULL",
         "FK to orders"),
        ("review_score",
         "review_score BETWEEN 1 AND 5",
         "Valid review score — NULL từ TRY_CAST cũng bị drop"),
    ],
    warn_rules=[
        ("response_time_hours",
         "response_time_hours >= 0 OR response_time_hours IS NULL",
         "Response trước review — suspicious"),
    ],
    dedup_keys=["review_id"]
)

(silver_reviews.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.reviews"))

count = spark.read.table(f"{CATALOG}.silver.reviews").count()
log("SUCCESS", "silver_reviews",
    f"Written to {CATALOG}.silver.reviews",
    {"rows": f"{count:,}"})

print("\n=== Sentiment distribution ===")
spark.sql(f"""
    SELECT sentiment,
           COUNT(*)                           AS count,
           ROUND(AVG(review_score), 2)        AS avg_score,
           SUM(CASE WHEN has_comment
               THEN 1 ELSE 0 END)             AS with_comment,
           ROUND(AVG(response_time_hours), 1) AS avg_response_hrs
    FROM {CATALOG}.silver.reviews
    GROUP BY sentiment
    ORDER BY avg_score
""").show()

In [0]:
log("INFO", "silver_products", "Starting transform")

bronze_products = spark.read.table(f"{CATALOG}.bronze.products")

bronze_categories = (spark.read.table(f"{CATALOG}.bronze.category_map")
    .select("product_category_name", "product_category_name_english"))

silver_products = (bronze_products
    .join(bronze_categories, "product_category_name", "left")
    .withColumn("product_name_lenght",
        expr("try_cast(product_name_lenght AS int)"))
    .withColumn("product_description_lenght",
        expr("try_cast(product_description_lenght AS int)"))
    .withColumn("product_photos_qty",
        expr("try_cast(product_photos_qty AS int)"))
    .withColumn("product_weight_g",
        expr("try_cast(product_weight_g AS double)"))
    .withColumn("product_length_cm",
        expr("try_cast(product_length_cm AS double)"))
    .withColumn("product_height_cm",
        expr("try_cast(product_height_cm AS double)"))
    .withColumn("product_width_cm",
        expr("try_cast(product_width_cm AS double)"))

    # Null strategy
    .withColumn("product_category_name_english",
        coalesce(
            col("product_category_name_english"),
            col("product_category_name"),
            lit("uncategorized")))

    # Unit conversion
    .withColumn("weight_kg",
        (col("product_weight_g") / lit(1000.0)).cast("double"))
    .withColumn("volume_cm3",
        (col("product_length_cm") *
         col("product_height_cm") *
         col("product_width_cm")).cast("double"))

    # Size tier
    .withColumn("size_tier",
        when(col("weight_kg").isNull(),   "unknown")
        .when(col("weight_kg") == 0,      "digital")
        .when(col("weight_kg") <= 0.3,    "XS")
        .when(col("weight_kg") <= 1.0,    "S")
        .when(col("weight_kg") <= 5.0,    "M")
        .when(col("weight_kg") <= 20.0,   "L")
        .otherwise("XL"))

    # Listing quality score (coalesce trước khi so sánh, tránh NULL lan ra)
    .withColumn("listing_quality_score",
        when(coalesce(col("product_photos_qty"), lit(0)) >= 5, lit(5))
        .when(coalesce(col("product_photos_qty"), lit(0)) >= 3, lit(3))
        .when(coalesce(col("product_photos_qty"), lit(0)) >= 1, lit(1))
        .otherwise(lit(0)) +
        when(coalesce(col("product_description_lenght"), lit(0)) > 500, lit(3))
        .when(coalesce(col("product_description_lenght"), lit(0)) > 100, lit(2))
        .when(coalesce(col("product_description_lenght"), lit(0)) > 0,   lit(1))
        .otherwise(lit(0)))

    .drop("_source_file", "_batch_id", "_ingested_at")
)

silver_products = dq_check(
    silver_products, "silver_products",
    critical_rules=[
        ("product_id",
         "product_id IS NOT NULL",
         "Primary key"),
    ],
    warn_rules=[
        ("product_weight_g",
         "product_weight_g > 0 OR product_weight_g IS NULL",
         "Weight âm: impossible"),
        ("product_photos_qty",
         "product_photos_qty > 0",
         "Không có ảnh: listing quality kém"),
    ],
    dedup_keys=["product_id"]
)

(silver_products.write
    .format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.products"))

count = spark.read.table(f"{CATALOG}.silver.products").count()
log("SUCCESS", "silver_products",
    f"Written to {CATALOG}.silver.products",
    {"rows": f"{count:,}"})

print("\n=== Top 10 categories ===")
spark.sql(f"""
    SELECT product_category_name_english AS category,
           COUNT(*)                      AS products,
           ROUND(AVG(weight_kg), 2)      AS avg_weight_kg,
           size_tier
    FROM {CATALOG}.silver.products
    GROUP BY product_category_name_english, size_tier
    ORDER BY products DESC
    LIMIT 10
""").show()

In [0]:
log("INFO", "silver_customers_sellers", "Starting transform")
bronze_customers = spark.read.table(f"{CATALOG}.bronze.customers")
bronze_sellers   = spark.read.table(f"{CATALOG}.bronze.sellers")
bronze_geo       = spark.read.table(f"{CATALOG}.bronze.geolocation")

# Geo lookup — average per zip code
geo_avg = (bronze_geo
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        avg("geolocation_lat").alias("geo_lat"),
        avg("geolocation_lng").alias("geo_lng")
    ))

# Brazil region mapping
def add_brazil_region(df, state_col):
    return df.withColumn("brazil_region",
        when(col(state_col).isin(
            ["AM","PA","AC","RO","RR","AP","TO"]),   "Norte")
        .when(col(state_col).isin(
            ["MA","PI","CE","RN","PB","PE",
             "AL","SE","BA"]),                        "Nordeste")
        .when(col(state_col).isin(
            ["MT","MS","GO","DF"]),                   "Centro-Oeste")
        .when(col(state_col).isin(
            ["SP","RJ","MG","ES"]),                   "Sudeste")
        .when(col(state_col).isin(
            ["PR","SC","RS"]),                        "Sul")
        .otherwise("Unknown"))

# Silver Customers
silver_customers = (bronze_customers
    .dropDuplicates(["customer_unique_id"])
    .withColumn("customer_city",
        initcap(lower(trim(col("customer_city")))))
    .withColumn("customer_state",
        upper(trim(col("customer_state"))))
    .join(geo_avg,
        col("customer_zip_code_prefix") ==
        col("geolocation_zip_code_prefix"), "left")
    .drop("geolocation_zip_code_prefix",
          "_source_file", "_batch_id")
)
silver_customers = add_brazil_region(silver_customers, "customer_state")

silver_customers = dq_check(
    silver_customers, "silver_customers",
    critical_rules=[
        ("customer_unique_id",
         "customer_unique_id IS NOT NULL",
         "Primary key thật của customer"),
        ("customer_state",
         "customer_state IS NOT NULL",
         "State required cho regional analysis"),
    ],
    warn_rules=[
        ("geo_lat",
         "geo_lat IS NOT NULL",
         "Không có coordinates — zip không match geo table"),
    ]
)

(silver_customers.write
    .format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.customers"))

# Silver Sellers
silver_sellers = (bronze_sellers
    .dropDuplicates(["seller_id"])
    .withColumn("seller_city",
        initcap(lower(trim(col("seller_city")))))
    .withColumn("seller_state",
        upper(trim(col("seller_state"))))
    .join(geo_avg,
        col("seller_zip_code_prefix") ==
        col("geolocation_zip_code_prefix"), "left")
    .drop("geolocation_zip_code_prefix",
          "_source_file", "_batch_id")
)
silver_sellers = add_brazil_region(silver_sellers, "seller_state")

silver_sellers = dq_check(
    silver_sellers, "silver_sellers",
    critical_rules=[
        ("seller_id",
         "seller_id IS NOT NULL",
         "Primary key"),
        ("seller_state",
         "seller_state IS NOT NULL",
         "State required"),
    ]
)

(silver_sellers.write
    .format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.silver.sellers"))

log("SUCCESS", "silver_customers_sellers",
    "Both tables written",
    {"customers": f"{silver_customers.count():,}",
     "sellers":   f"{silver_sellers.count():,}"})

print("\n=== Customers by region ===")
spark.sql(f"""
    SELECT brazil_region,
           COUNT(*) AS customers
    FROM {CATALOG}.silver.customers
    GROUP BY brazil_region
    ORDER BY customers DESC
""").show()

In [0]:
spark.sql("""
    SELECT * FROM (
        SELECT 'orders'      AS tbl,
               COUNT(*)      AS rows
        FROM olist_ecommerce.silver.orders      UNION ALL
        SELECT 'order_items', COUNT(*)
        FROM olist_ecommerce.silver.order_items UNION ALL
        SELECT 'payment_agg', COUNT(*)
        FROM olist_ecommerce.silver.payment_agg UNION ALL
        SELECT 'reviews',     COUNT(*)
        FROM olist_ecommerce.silver.reviews     UNION ALL
        SELECT 'products',    COUNT(*)
        FROM olist_ecommerce.silver.products    UNION ALL
        SELECT 'customers',   COUNT(*)
        FROM olist_ecommerce.silver.customers   UNION ALL
        SELECT 'sellers',     COUNT(*)
        FROM olist_ecommerce.silver.sellers
    )
    ORDER BY rows DESC
""").show()